In [1]:
import sys
sys.path.append('../')

import importlib
import pandas as pd
import numpy as np
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

import src.display as _display
importlib.reload(_display)
from src.display import (
    init_notebook_theme, show_hero, show_section, show_insight,
    show_metrics_row, show_findings_list, show_badge, show_table_html, show_architecture_card,
)
init_notebook_theme()

show_hero(
    title="Partie commune — Architecture MLOps",
    objective="Industrialiser les modèles fraude et segmentation.",
    plan_items=["Pipeline données", "Versionning MLflow", "Déploiement Docker",
                "Monitoring dérive", "CI/CD GitHub Actions"],
)
show_section("1. Pipeline de données")
show_architecture_card("Pipeline", [
    "Source", "Ingestion", "Validation GE", "Nettoyage", "Features", "Split", "MLflow",
])


In [2]:
# Pipeline de validation avec Great Expectations
# Exemple de règles de validation pour detection_fraude.csv

FRAUD_SCHEMA = {
    "required_columns": ["step", "type", "amount", "nameOrig", "oldbalanceOrg",
                          "newbalanceOrig", "nameDest", "oldbalanceDest",
                          "newbalanceDest", "isFraud"],
    "type_checks": {
        "amount": "float",
        "isFraud": "int",
    },
    "value_checks": {
        "amount": {"min": 0},
        "isFraud": {"allowed": [0, 1]},
        "type": {"allowed": ["PAYMENT", "TRANSFER", "CASH_OUT", "DEBIT", "CASH_IN"]},
    }
}


def validate_dataset(df: pd.DataFrame, schema: dict) -> dict:
    """Validation légère du dataset avant entraînement."""
    report = {"passed": True, "errors": []}

    # Colonnes manquantes
    missing = set(schema["required_columns"]) - set(df.columns)
    if missing:
        report["errors"].append(f"Colonnes manquantes : {missing}")
        report["passed"] = False

    # Vérifications de valeurs
    for col, checks in schema.get("value_checks", {}).items():
        if col not in df.columns:
            continue
        if "min" in checks and (df[col] < checks["min"]).any():
            n = (df[col] < checks["min"]).sum()
            report["errors"].append(f"{col} : {n} valeurs < {checks['min']}")
        if "allowed" in checks:
            invalid = ~df[col].isin(checks["allowed"])
            if invalid.any():
                report["errors"].append(f"{col} : valeurs invalides {df[col][invalid].unique()[:5]}")

    # Valeurs manquantes
    nulls = df[schema["required_columns"]].isnull().sum()
    if nulls.any():
        report["warnings"] = f"Valeurs manquantes : {nulls[nulls > 0].to_dict()}"

    return report


# Test sur un sample
sample = pd.DataFrame({
    "step": [1], "type": ["PAYMENT"], "amount": [100.0],
    "nameOrig": ["C123"], "oldbalanceOrg": [500.0], "newbalanceOrig": [400.0],
    "nameDest": ["M456"], "oldbalanceDest": [0.0], "newbalanceDest": [100.0],
    "isFraud": [0]
})

result = validate_dataset(sample, FRAUD_SCHEMA)
print("Validation :" , "✓ OK" if result["passed"] else "✗ ERREURS")
if result["errors"]:
    for e in result["errors"]:
        print(f"  - {e}")

Validation : ✓ OK


In [3]:
show_section("2. Versionning avec MLflow")

In [4]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

# Configuration MLflow (local) — SQLite requis depuis MLflow 3.x
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("fraud_detection")

# Simulation d'un run MLflow (stratify + assez d'échantillons pour évaluer le F1)
X_demo, y_demo = make_classification(n_samples=5000, n_features=10,
                                      weights=[0.99, 0.01], random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_demo, y_demo, test_size=0.2, random_state=42, stratify=y_demo,
)

with mlflow.start_run(run_name="random_forest_v1"):
    params = {"n_estimators": 100, "max_depth": 10, "class_weight": "balanced"}
    mlflow.log_params(params)

    model = RandomForestClassifier(**params, random_state=42)
    model.fit(X_tr, y_tr)

    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    metrics = {
        "roc_auc": round(roc_auc_score(y_te, y_proba), 4),
        "f1": round(f1_score(y_te, y_pred), 4),
    }
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(model, name="random_forest")

    print("Run MLflow enregistré :")
    for k, v in metrics.items():
        print(f"  {k}: {v}")

2026/06/09 10:34:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run MLflow enregistré :
  roc_auc: 0.6886
  f1: 0.3636


In [5]:
show_section("3. Architecture de déploiement")
show_architecture_card("Prédiction", ["Client JSON", "FastAPI :8000", "Model Registry", "Réponse"])
show_architecture_card("Infrastructure", ["GitHub", "MLflow UI", "React (Render Static)", "FastAPI (Render)"])
show_insight("Implémentation : mlops/api/app.py, render.yaml (API + frontend)")

In [6]:
show_section("4. Monitoring & Détection de dérive")

In [7]:
from scipy import stats

def detect_data_drift(ref_data, new_data, threshold_psi=0.2):
    bins = np.percentile(ref_data, np.linspace(0, 100, 11))
    bins[0] -= 1e-9; bins[-1] += 1e-9
    ref_pct = np.histogram(ref_data, bins=bins)[0] / len(ref_data)
    new_pct = np.histogram(new_data, bins=bins)[0] / len(new_data)
    ref_pct = np.where(ref_pct == 0, 1e-6, ref_pct)
    new_pct = np.where(new_pct == 0, 1e-6, new_pct)
    psi = np.sum((new_pct - ref_pct) * np.log(new_pct / ref_pct))
    ks_stat, ks_pvalue = stats.ks_2samp(ref_data, new_data)
    status = "stable" if psi < 0.1 else ("warning" if psi < threshold_psi else "drift")
    return {"psi": round(psi, 4), "ks_stat": round(ks_stat, 4), "ks_pvalue": round(ks_pvalue, 4), "status": status}

np.random.seed(42)
ref = pd.Series(np.random.normal(1000, 500, 5000))
new_stable = pd.Series(np.random.normal(1050, 520, 1000))
new_drift = pd.Series(np.random.normal(2000, 800, 1000))
for label, data in [("Stables", new_stable), ("Dérive", new_drift)]:
    r = detect_data_drift(ref, data)
    show_badge(f"{label} : {r['status'].upper()}", r["status"])
    show_metrics_row({"PSI": r["psi"], "KS": r["ks_stat"]})

In [8]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import importlib
import src.utils as _utils
importlib.reload(_utils)
from src.utils import COLORS, show_figure  # sys.path configuré en cellule 2

weeks = list(range(1, 25))
np.random.seed(0)
perf_stable = 0.95 + np.random.normal(0, 0.005, 24)
perf_drift = np.concatenate([0.95 + np.random.normal(0, 0.005, 12),
                               np.linspace(0.95, 0.80, 12) + np.random.normal(0, 0.01, 12)])
psi_values = np.concatenate([np.random.uniform(0.02, 0.08, 12), np.linspace(0.08, 0.35, 12)])
psi_colors = ['green' if p < 0.1 else 'orange' if p < 0.2 else 'red' for p in psi_values]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Distribution : référence vs nouvelles données',
                    'Évolution ROC-AUC dans le temps', 'PSI (Population Stability Index)'],
)

fig.add_trace(go.Histogram(x=ref, name='Référence', opacity=0.6, marker_color=COLORS['primary'], histnorm='probability density'), row=1, col=1)
fig.add_trace(go.Histogram(x=new_drift, name='Nouvelles données', opacity=0.6, marker_color=COLORS['secondary'], histnorm='probability density'), row=1, col=1)
fig.add_trace(go.Scatter(x=weeks, y=perf_stable, mode='lines+markers', name='Stable', line_color=COLORS['primary']), row=1, col=2)
fig.add_trace(go.Scatter(x=weeks, y=perf_drift, mode='lines+markers', name='Avec dérive', line_color=COLORS['secondary']), row=1, col=2)
fig.add_hline(y=0.90, line_dash='dash', line_color='orange', annotation_text='Seuil alerte', row=1, col=2)
fig.add_hline(y=0.85, line_dash='dash', line_color='red', annotation_text='Seuil critique', row=1, col=2)
fig.add_trace(go.Bar(x=weeks, y=psi_values, marker_color=psi_colors, showlegend=False), row=1, col=3)
fig.add_hline(y=0.1, line_dash='dash', line_color='orange', annotation_text='Warning (0.1)', row=1, col=3)
fig.add_hline(y=0.2, line_dash='dash', line_color='red', annotation_text='Drift (0.2)', row=1, col=3)

fig.update_layout(title='Dashboard de monitoring MLOps', barmode='overlay', height=450, width=1300)
fig.update_xaxes(title_text='Semaine', row=1, col=2)
fig.update_xaxes(title_text='Semaine', row=1, col=3)
show_figure(fig, '../reports/figures/mlops_monitoring.png', width=1300, height=450)


In [9]:
show_section("5. CI/CD avec GitHub Actions")
show_architecture_card("CI/CD", ["Push/PR", "Tests", "Train", "Validate", "Docker", "Deploy"])
show_insight("Voir .github/workflows/ml_pipeline.yml")

In [10]:
show_section("Synthèse MLOps")
mlops_df = pd.DataFrame([
    {"Composant": "Données", "Outil": "DVC", "Rôle": "Versionning CSV"},
    {"Composant": "Modèles", "Outil": "MLflow", "Rôle": "Registre expériences"},
    {"Composant": "Qualité", "Outil": "Great Expectations", "Rôle": "Contrats données"},
    {"Composant": "API", "Outil": "FastAPI", "Rôle": "/predict/fraud & /segment"},
    {"Composant": "Monitoring", "Outil": "PSI + KS", "Rôle": "Dérive"},
    {"Composant": "CI/CD", "Outil": "GitHub Actions", "Rôle": "Deploy auto"},
])
show_table_html(mlops_df.set_index("Composant"), title="Stack MLOps")
show_findings_list([
    "Fraude : réentraînement si ROC-AUC < 0.90 ou PSI > 0.2",
    "Clustering : recalcul mensuel",
    "Deploy si métriques > baseline − 2%",
], title="Politique de réentraînement")

,Outil,Rôle
Composant,,
Données,DVC,Versionning CSV
Modèles,MLflow,Registre expériences
Qualité,Great Expectations,Contrats données
API,FastAPI,/predict/fraud & /segment
Monitoring,PSI + KS,Dérive
CI/CD,GitHub Actions,Deploy auto
